# Diff-Img2Img: Low-Light Enhancement with Retinex-Diffusion

This notebook provides a step-by-step guide to:
1. **Data Synthesis** — Generate low-light images from normal-light datasets
2. **Training** — Train the Retinex-Diffusion model
3. **Inference** — Enhance low-light images
4. **Evaluation** — Compute PSNR, SSIM, LPIPS metrics

## 0. Environment Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -r ../requirements.txt

import os, sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Dataset Preparation

### 1.1 Expected Dataset Structure

```
your_dataset/
├── our485/          # Training (485 pairs)
│   ├── high/        # Normal-light ground truth
│   └── low/         # Low-light (auto-generated or real)
└── eval15/          # Test (15 pairs)
    ├── high/
    └── low/
```

> **Note**: With `online_synthesis=True`, the `low/` directory is optional — degradation is applied on-the-fly during training.

In [ ]:
# ==============================
# ⬇️  SET YOUR DATASET PATH HERE
# ==============================
DATA_DIR = "/mnt/f/datasets/kitti_LOL"  # Change this to your dataset path

assert os.path.exists(DATA_DIR), f"Dataset not found: {DATA_DIR}"
print("Train images:", len(os.listdir(os.path.join(DATA_DIR, "our485", "high"))))
print("Test images: ", len(os.listdir(os.path.join(DATA_DIR, "eval15", "high"))))

### 1.2 Synthesize Low-Light Data with Darker Engine

The `Darker` engine applies physics-based degradation:
- **Gamma correction** + linear attenuation (global darkening)
- **Poisson-Gaussian noise** (realistic sensor model)
- **Random multi-source headlights** (local illumination)
- **Vignetting**, **motion blur**, **JPEG artifacts** (optional)

In [ ]:
from scripts.darker import Darker
import cv2

# Create Darker engine with randomized parameters
darker = Darker(
    data_dir=DATA_DIR,
    phase="train",
    randomize=True,
    param_ranges={
        "gamma": (1.5, 4.0),
        "linear_attenuation": (0.25, 0.7),
    }
)

# Demo: degrade a single image
sample_path = os.path.join(DATA_DIR, "our485", "high")
sample_file = sorted(os.listdir(sample_path))[0]
img = cv2.imread(os.path.join(sample_path, sample_file))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (Normal Light)")

for i in range(1, 4):
    degraded = darker.degrade_single(img)  # Each call generates different degradation
    axes[i].imshow(cv2.cvtColor(degraded, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Degraded #{i}")

for ax in axes: ax.axis("off")
plt.suptitle("Randomized Degradation Examples", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: Batch synthesize the entire dataset (only needed if NOT using online_synthesis)
# darker.process_images()  # Uncomment to run

### 1.3 Visualize the Dataset

In [ ]:
from datasets.data_set import LowLightDataset
from torch.utils.data import DataLoader

# Create dataset with online synthesis enabled
train_dataset = LowLightDataset(
    image_dir=DATA_DIR, img_size=256, phase="train",
    online_synthesis=True
)
print(f"Training samples: {len(train_dataset)}")

# Visualize a batch
loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
low_batch, high_batch = next(iter(loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    low_np = (low_batch[i].permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1)
    high_np = (high_batch[i].permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1)
    axes[0, i].imshow(low_np); axes[0, i].set_title(f"Low #{i+1}"); axes[0, i].axis("off")
    axes[1, i].imshow(high_np); axes[1, i].set_title(f"GT #{i+1}"); axes[1, i].axis("off")
plt.suptitle("Training Batch (Online Synthesis)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Model Architecture

The pipeline consists of two models:

| Model | Purpose | Architecture |
|-------|---------|-------------|
| **DecomNet** | Retinex decomposition → Reflectance + Illumination | Lightweight CNN (C2f blocks) |
| **UNet2D** | Conditional diffusion denoising | HuggingFace Diffusers UNet2DModel |

### Loss Functions

| Loss | Weight | Function |
|------|--------|----------|
| Diffusion (Charbonnier) | 1.0 | Min-SNR weighted noise prediction |
| Charbonnier (pixel) | 1.0 | Robust L1 approximation on pred_x0 |
| SSIM (structure) | 0.1 | Structural similarity |
| LPIPS (perceptual) | 0.1 | VGG-based perceptual quality |
| Retinex (recon+refl+TV) | 0.1 | Decompositoin consistency |

In [ ]:
from diffusers import UNet2DModel
from models.retinex import DecomNet
from models.diffusion import CombinedModel

# Initialize models
unet = UNet2DModel(
    sample_size=256,
    in_channels=9,   # 3 (noisy) + 3 (reflectance) + 3 (illumination)
    out_channels=3,
    layers_per_block=2,
    block_out_channels=[32, 64, 128, 256, 512],
    down_block_types=["DownBlock2D"] * 5,
    up_block_types=["UpBlock2D"] * 5,
)

decom = DecomNet()
combined = CombinedModel(unet, decom)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"UNet parameters:     {count_params(unet):>12,}")
print(f"DecomNet parameters: {count_params(decom):>12,}")
print(f"Total parameters:    {count_params(combined):>12,}")

## 3. Training

### 3.1 Launch Training via CLI

Training uses `accelerate` for mixed precision and multi-GPU support. Run from terminal:

```bash
accelerate launch main.py --mode train \
    --data_dir "/mnt/f/datasets/kitti_LOL" \
    --output_dir "runs/retinex_exp" \
    --use_retinex \
    --online_synthesis \
    --freeze_decom_steps 2000 \
    --epochs 100 \
    --batch_size 4 \
    --resolution 256 \
    --mixed_precision fp16 \
    --prediction_type v_prediction \
    --gradient_accumulation_steps 4
```

### 3.2 Launch from Notebook

In [ ]:
# Option A: Run directly in subprocess (non-blocking)
import subprocess

OUTPUT_DIR = "runs/notebook_exp"

cmd = f"""
accelerate launch main.py --mode train \
    --data_dir "{DATA_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --use_retinex \
    --online_synthesis \
    --freeze_decom_steps 2000 \
    --epochs 50 \
    --batch_size 4 \
    --resolution 256 \
    --mixed_precision fp16 \
    --prediction_type v_prediction \
    --gradient_accumulation_steps 4
""".strip()

print("Command:")
print(cmd)
print("\n⚠️ Uncomment the line below to actually start training:")
# !{cmd}

In [ ]:
# Option B: Use the DiffusionEngine directly (for custom control)
# from core.engine import DiffusionEngine
# import argparse
# 
# args = argparse.Namespace(
#     mode="train", data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
#     use_retinex=True, online_synthesis=True, freeze_decom_steps=2000,
#     epochs=50, batch_size=4, resolution=256, seed=42,
#     mixed_precision="fp16", prediction_type="v_prediction",
#     # ... add all other required args from main.py get_args()
# )
# engine = DiffusionEngine(args)
# engine.train()

### 3.3 Monitor Training

Training logs are written to TensorBoard. Launch in a separate terminal:

```bash
tensorboard --logdir runs/notebook_exp/logs
```

In [ ]:
# Load and plot training metrics (if training has completed/is running)
import pandas as pd

csv_path = os.path.join(OUTPUT_DIR, "training_metrics.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(df["step"], df["loss"], alpha=0.7)
    ax1.set_title("Training Loss"); ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
    ax2.plot(df["step"], df["lr"], color="orange")
    ax2.set_title("Learning Rate"); ax2.set_xlabel("Step"); ax2.set_ylabel("LR")
    plt.tight_layout(); plt.show()
else:
    print(f"No training metrics found at {csv_path}. Run training first.")

## 4. Inference (Enhance Low-Light Images)

Load a trained model and enhance low-light images.

In [ ]:
from scripts.visual_val import load_models, run_inference, tensor_to_pil
from torchvision import transforms
from PIL import Image

# ==============================
# ⬇️  SET YOUR MODEL PATH HERE
# ==============================
MODEL_PATH = "runs/notebook_exp"  # Path to trained model
USE_RETINEX = True

# Load models
try:
    models = load_models(MODEL_PATH, USE_RETINEX, device)
    print("✅ Models loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load models: {e}")
    print("Make sure training has completed and the model is saved.")

In [ ]:
# Enhance a test image
test_low_dir = os.path.join(DATA_DIR, "eval15", "low")
test_high_dir = os.path.join(DATA_DIR, "eval15", "high")

if os.path.exists(test_low_dir):
    test_files = sorted([f for f in os.listdir(test_low_dir) if f.endswith(('.png', '.jpg'))])
    
    tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    # Process first 4 images
    n_show = min(4, len(test_files))
    fig, axes = plt.subplots(3, n_show, figsize=(4*n_show, 12))
    
    for i in range(n_show):
        # Load
        img_low = Image.open(os.path.join(test_low_dir, test_files[i])).convert("RGB")
        img_high = Image.open(os.path.join(test_high_dir, test_files[i])).convert("RGB")
        t_low = tf(img_low)
        
        # Enhance
        with torch.no_grad():
            t_out = run_inference(models, t_low, num_inference_steps=20, device=device)
        img_out = tensor_to_pil(t_out)
        
        # Display
        axes[0, i].imshow(img_low.resize((256, 256))); axes[0, i].set_title("Input (Low)"); axes[0, i].axis("off")
        axes[1, i].imshow(img_out); axes[1, i].set_title("Enhanced"); axes[1, i].axis("off")
        axes[2, i].imshow(img_high.resize((256, 256))); axes[2, i].set_title("Ground Truth"); axes[2, i].axis("off")
    
    plt.suptitle("Low-Light Enhancement Results", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print(f"Test directory not found: {test_low_dir}")

## 5. Evaluation (PSNR, SSIM, LPIPS)

In [ ]:
from torcheval.metrics.functional import peak_signal_noise_ratio
from utils.misc import ssim

# Try LPIPS
try:
    import lpips
    lpips_fn = lpips.LPIPS(net='vgg', verbose=False).to(device)
    lpips_fn.eval()
    has_lpips = True
except ImportError:
    print("⚠️ lpips not installed. Run: pip install lpips")
    has_lpips = False

# Evaluate on test set
if os.path.exists(test_low_dir):
    psnr_vals, ssim_vals, lpips_vals = [], [], []
    
    for f in sorted(os.listdir(test_low_dir)):
        if not f.endswith(('.png', '.jpg')): continue
        
        img_low = Image.open(os.path.join(test_low_dir, f)).convert("RGB")
        img_high = Image.open(os.path.join(test_high_dir, f)).convert("RGB")
        
        t_low = tf(img_low)
        t_high = tf(img_high).unsqueeze(0).to(device)
        t_high_01 = (t_high / 2 + 0.5).clamp(0, 1)
        
        with torch.no_grad():
            t_out = run_inference(models, t_low, 20, device)
        
        # Compute metrics
        psnr_val = peak_signal_noise_ratio(t_out, t_high_01, data_range=1.0).item()
        ssim_val = ssim(t_out, t_high_01).item()
        psnr_vals.append(psnr_val)
        ssim_vals.append(ssim_val)
        
        if has_lpips:
            lpips_val = lpips_fn(t_out * 2 - 1, t_high).mean().item()
            lpips_vals.append(lpips_val)
        
        print(f"  {f}: PSNR={psnr_val:.2f}, SSIM={ssim_val:.4f}" + 
              (f", LPIPS={lpips_vals[-1]:.4f}" if has_lpips else ""))
    
    print("\n" + "="*50)
    print(f"Average PSNR:  {np.mean(psnr_vals):.2f} dB")
    print(f"Average SSIM:  {np.mean(ssim_vals):.4f}")
    if has_lpips:
        print(f"Average LPIPS: {np.mean(lpips_vals):.4f} (lower is better)")
    print("="*50)

## 6. Quick Reference

### CLI Commands

| Task | Command |
|------|--------|
| Train | `accelerate launch main.py --mode train --use_retinex --online_synthesis ...` |
| Predict (images) | `python main.py --mode predict --model_path runs/exp --use_retinex ...` |
| Predict (video) | `python main.py --mode predict --video_path input.mp4 --use_retinex ...` |
| Validate | `python main.py --mode validate --model_path runs/exp --use_retinex` |
| Web UI | `python main.py --mode ui` |

### Key Training Flags

| Flag | Default | Recommended |
|------|---------|-------------|
| `--online_synthesis` | off | ✅ Always on for training |
| `--use_retinex` | off | ✅ Better results |
| `--freeze_decom_steps` | 0 | 2000 (stabilize early training) |
| `--prediction_type` | v_prediction | Keep default |
| `--offset_noise` | off | Try for very dark images |